In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nafiasgrinder/wine-dataset/wine.csv


In [2]:
import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
import math

In [3]:
df = pd.read_csv("/kaggle/input/datasets/nafiasgrinder/wine-dataset/wine.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 178 entries, 0 to 177
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Wine                  178 non-null    int64  
 1   Alcohol               178 non-null    float64
 2   Malic.acid            178 non-null    float64
 3   Ash                   178 non-null    float64
 4   Acl                   178 non-null    float64
 5   Mg                    178 non-null    int64  
 6   Phenols               178 non-null    float64
 7   Flavanoids            178 non-null    float64
 8   Nonflavanoid.phenols  178 non-null    float64
 9   Proanth               178 non-null    float64
 10  Color.int             178 non-null    float64
 11  Hue                   178 non-null    float64
 12  OD                    178 non-null    float64
 13  Proline               178 non-null    int64  
dtypes: float64(11), int64(3)
memory usage: 19.6 KB


In [4]:
class WineDataset(Dataset):
    
    def __init__(self, transform=None):
        # data loading 
        xy = np.loadtxt('/kaggle/input/datasets/nafiasgrinder/wine-dataset/wine.csv', delimiter=',', dtype=np.float32, skiprows=1)
        self.n_samples = xy.shape[0]
        
        self.x = xy[:, 1:] # ilk sütun (şarabın türü yani y ) hariç
        self.y = xy[:,[0]]

        self.transform = transform
        
    def __getitem__(self, index):
        # dataset[index]
        sample = self.x[index], self.y[index]
        
        if self.transform:
            sample = self.transform(sample) 
        
        return sample

    def __len__(self):
        # len(dataset)
        return self.n_samples

In [5]:
class ToTensor:
    def __call__(self, sample):
        inputs, targets = sample
        return torch.from_numpy(inputs), torch.from_numpy(targets)

In [6]:
class MulTransform():
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, sample):
        inputs, target = sample
        inputs *= self.factor
        return inputs, target
    

In [7]:
dataset = WineDataset(transform=ToTensor())

In [8]:
first_data = dataset[0]
features, labels = first_data
print(features, labels)
type(features), type(labels)

tensor([1.4230e+01, 1.7100e+00, 2.4300e+00, 1.5600e+01, 1.2700e+02, 2.8000e+00,
        3.0600e+00, 2.8000e-01, 2.2900e+00, 5.6400e+00, 1.0400e+00, 3.9200e+00,
        1.0650e+03]) tensor([1.])


(torch.Tensor, torch.Tensor)

In [9]:
composed = torchvision.transforms.Compose([ToTensor(), MulTransform(2)])

In [10]:
dataset2 = WineDataset(transform=composed)

In [11]:
first_data = dataset2[0]
features, labels = first_data
print(features, labels)
type(features), type(labels)

tensor([2.8460e+01, 3.4200e+00, 4.8600e+00, 3.1200e+01, 2.5400e+02, 5.6000e+00,
        6.1200e+00, 5.6000e-01, 4.5800e+00, 1.1280e+01, 2.0800e+00, 7.8400e+00,
        2.1300e+03]) tensor([1.])


(torch.Tensor, torch.Tensor)

In [12]:
dataloader = DataLoader(dataset=dataset,batch_size=4,shuffle=True, num_workers=2)

In [13]:
# training loop
num_epochs = 2
total_samples = len(dataset)
n_iteration = math.ceil(total_samples / 4)
print(total_samples, n_iteration)

178 45


In [14]:
for epoch in range(num_epochs):
    for i, (input, labels) in enumerate(dataloader):
        # forward backward, update
        if (i + 1) % 5 == 0:
            print(f"epoc {epoch+1}/{num_epochs}, step {i+1}/{n_iteration}, inputs {input.shape}")

epoc 1/2, step 5/45, inputs torch.Size([4, 13])
epoc 1/2, step 10/45, inputs torch.Size([4, 13])
epoc 1/2, step 15/45, inputs torch.Size([4, 13])
epoc 1/2, step 20/45, inputs torch.Size([4, 13])
epoc 1/2, step 25/45, inputs torch.Size([4, 13])
epoc 1/2, step 30/45, inputs torch.Size([4, 13])
epoc 1/2, step 35/45, inputs torch.Size([4, 13])
epoc 1/2, step 40/45, inputs torch.Size([4, 13])
epoc 1/2, step 45/45, inputs torch.Size([2, 13])
epoc 2/2, step 5/45, inputs torch.Size([4, 13])
epoc 2/2, step 10/45, inputs torch.Size([4, 13])
epoc 2/2, step 15/45, inputs torch.Size([4, 13])
epoc 2/2, step 20/45, inputs torch.Size([4, 13])
epoc 2/2, step 25/45, inputs torch.Size([4, 13])
epoc 2/2, step 30/45, inputs torch.Size([4, 13])
epoc 2/2, step 35/45, inputs torch.Size([4, 13])
epoc 2/2, step 40/45, inputs torch.Size([4, 13])
epoc 2/2, step 45/45, inputs torch.Size([2, 13])
